In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
llm = ChatOllama(model="llama3.1", temperature=0)

# 1. Define what a "perfect" score looks like using Pydantic
# This forces the Judge LLM to reply with a structured JSON score and explanation
class EvaluationScore(BaseModel):
    score: int = Field(description="Score from 1 to 5, where 5 is perfect.")
    reasoning: str = Field(description="A short explanation of why this score was given.")

# Bind this structured output rule to our Judge model -> you use the evaluation score as the blueprint for this output
    # The Translation: LangChain takes your Python Pydantic class and translates it into a strict JSON Schema block.
    # The Injection: It takes that JSON Schema and injects it into the LLM's hidden system prompt, making it only reply in this format
    # The Interception: When the LLM generates its response, it spits out a raw JSON string.
    # The Validation: LangChain catches that string, feeds it back into your Pydantic bouncer to 
        # ensure the LLM didn't output the wrong things
    # The Hand-off: If it passes, LangChain converts that JSON back into a neat Python object and hands it to you.
judge_llm = llm.with_structured_output(EvaluationScore)

# 2. The Grading Rubric (The Prompt)
judge_prompt = ChatPromptTemplate.from_template("""
You are an impartial, expert evaluator. Grade the Agent's performance based on the transcript below.

Rubric:
5: Perfect. Used tools correctly, no hallucinations, gave a direct answer.
3: Okay. Answered the question but hallucinated a tool or took unnecessary steps.
1: Fail. Did not answer the question or completely misused tools.

User Query: {query}
Agent Transcript: {transcript}

Analyze the transcript and provide a score and reasoning.
""")

# 3. Create our Judge Pipeline
    # | is LCEL (LangChain Expression Language) -> | symbol means "Pipe" or "Chain."
        # evaluator = prompt | llm | database_saver
    # prompt on left side, llm on right side
evaluator = judge_prompt | judge_llm

print("Module 4, Step 1 & 2 Complete: Judge LLM is primed and ready with a rubric!")

Module 4, Step 1 & 2 Complete: Judge LLM is primed and ready with a rubric!


Running an eval pipeline

In [3]:
# 1. Define our test data
test_query = "What is 5 plus 5?"

# We simulate a transcript where the agent did the right thing, 
# but maybe talked a bit too much or acted slightly confused first.
fake_transcript = """
Turn 1 - Agent Thought: I should probably search the web for this.
Turn 1 - Agent Action: web_search("what is 5 plus 5")
Turn 1 - Tool Result: Error: web_search tool not found.
Turn 2 - Agent Thought: Oops, I don't have a web search tool. I will use the calculator.
Turn 2 - Agent Action: multiply(a=5, b=2) 
Turn 2 - Tool Result: 10
Turn 3 - Agent Thought: Wait, the user asked for addition, not multiplication. But 5x2 is 10, and 5+5 is 10.
Turn 3 - Agent Final Answer: The answer is 10.
"""

print("Asking the Judge to evaluate the transcript...\n")

# 2. Run the evaluation pipeline
try:
    evaluation = evaluator.invoke({"query": test_query, "transcript": fake_transcript})
    
    # 3. Print the structured results
    print("⚖️  EVALUATION RESULTS  ⚖️")
    print(f"Score: {evaluation.score} / 5")
    print(f"Reasoning: {evaluation.reasoning}")

except Exception as e:
    print(f"Evaluation failed. Error: {e}")
    print("\nNote: Smaller local models sometimes struggle to format strict JSON for evaluations. If this failed, a 70B model would handle it perfectly.")

print("\nModule 4 Complete! You now have an automated grading system.")

Asking the Judge to evaluate the transcript...

⚖️  EVALUATION RESULTS  ⚖️
Score: 1 / 5
Reasoning: The agent failed to directly address the user's query. Instead of using the correct tool for arithmetic operations (addition), it attempted to use an unknown web search tool and then incorrectly used multiplication instead of addition. The final answer was also incorrect.

Module 4 Complete! You now have an automated grading system.
